# PDF-Powered Telegram Chatbot (RAG with Gemma) + Customer Requests

This notebook builds a Telegram chatbot that:
1. Answers questions using the content of **PDF 1** (a text/narrative document,  via Retrieval-Augmented Generation (RAG) with the **Gemma-2b-it** model.
2. Handles **`/request`** — walks a customer through name → item → quantity, matching each item against categories from **PDF 2** (your data table document).

**Run the cells in order.** You'll be prompted to upload your two PDFs partway through.


## 1. Setup — install libraries

Colab already ships with `torch`, `numpy`, and `pandas` preinstalled and matched to its own
CUDA/runtime setup — reinstalling them (or letting another package silently upgrade them)
is the #1 cause of import errors in Colab. So here we:
- Only install what's actually missing.
- Pin `huggingface_hub` explicitly, since newer `sentence-transformers` / `transformers`
  releases can otherwise pull in a `huggingface_hub` version that breaks imports
  (e.g. `ImportError: cannot import name 'cached_download'`).
- **Restart the runtime once, right after installing** (Colab requires this any time
  package versions change) — the cell below does this for you automatically.


In [ ]:
!pip install -q -U pdfplumber PyPDF2
!pip install -q -U pyTelegramBotAPI
!pip install -q -U openpyxl
!pip install -q -U transformers accelerate sentencepiece
!pip install -q -U openpyxl pandas
!pip install -q -U sentence-transformers faiss-cpu

# Colab needs a fresh process after installing/upgrading packages like this.
# This restarts the runtime automatically — just re-run the notebook from the next
# cell onwards after it restarts (you do NOT need to re-run the pip installs above).
import os
os.kill(os.getpid(), 9)


**After the cell above runs, Colab's runtime will restart automatically** (you may see
a brief "Session crashed" message — that's expected, not an error). Once it's back,
continue running from the next cell onward. You do **not** need to re-run the install cell.


## 2. Imports

In [ ]:
import os
import json
import re
import torch
import pandas as pd
import numpy as np
import pdfplumber

from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss

from google.colab import userdata, files

print("Torch CUDA available:", torch.cuda.is_available())


## 3. API Keys

We pull the Hugging Face and Telegram tokens from **Colab Secrets**.

To add them: click the 🔑 key icon in the left sidebar of Colab → **Add new secret** → create:
- `HF` → your Hugging Face access token
- `telegram` → your Telegram bot token (from @BotFather)

Make sure the toggle "Notebook access" is turned ON for both.


In [ ]:
hf_key = userdata.get('HF')
telegram_token = userdata.get('telegram')
print("Keys loaded:", bool(hf_key), bool(telegram_token))


## 4. Authenticate with Hugging Face (needed to download Gemma)

In [ ]:
!huggingface-cli login --token {hf_key} --add-to-git-credential


**Important:** before this will work, you must accept Gemma's license once at
https://huggingface.co/google/gemma-2b-it (click "Agree and access repository"),
using the same account that generated your HF token.


## 5. Load the Gemma model

In [ ]:
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it", token=hf_key)

model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b-it",
    device_map="cuda",
    torch_dtype="auto",
    token=hf_key
)
print("Gemma loaded.")


## 6. Upload your two PDFs

Run the cell below, then use the "Choose Files" button to upload **both PDFs at once**
(select both files in the file picker):
narrative agreement, used for chat Q&A.
 — the data-table document, used to match categories for `/request`.


In [ ]:
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))


In [ ]:
# Filenames are hardcoded to match your actual files.
# If you rename the files before uploading, update these two lines to match.
SLA_PDF_PATH = "PDF1.pdf"          # narrative PDF (for chat)
DATA_PDF_PATH = "PDF2.pdf"  # table PDF (for /request category matching)

assert SLA_PDF_PATH in uploaded, f"{SLA_PDF_PATH} not found in uploaded files: {list(uploaded.keys())}"
assert DATA_PDF_PATH in uploaded, f"{DATA_PDF_PATH} not found in uploaded files: {list(uploaded.keys())}"

print("SLA_PDF_PATH  ->", SLA_PDF_PATH)
print("DATA_PDF_PATH ->", DATA_PDF_PATH)

## 7. Extract text from PDF 1 (for RAG chat)

We pull all the text out of PDF 1 and split it into overlapping chunks, so the
retrieval step can find the most relevant passage for any question.


In [ ]:
def extract_text_from_pdf(path):
    full_text = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text.append(text)
    return "\n".join(full_text)


def chunk_text(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


sla_text = extract_text_from_pdf(SLA_PDF_PATH)
sla_chunks = chunk_text(sla_text)
print(f"Extracted {len(sla_chunks)} chunks from {SLA_PDF_PATH}")


## 8. Build embeddings + FAISS index for retrieval

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedder.encode(sla_chunks, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("FAISS index built with", index.ntotal, "vectors.")


## 9. Retrieval function — find the most relevant chunks for a question

In [ ]:
def retrieve_relevant_chunks(question, top_k=3):
    query_embedding = embedder.encode([question]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return [sla_chunks[i] for i in indices[0] if i < len(sla_chunks)]


In [ ]:
def _dedupe_lines(values, tol=10):
    """Merges near-identical coordinate values (within `tol` points) into one,
    so a stray thin rectangle a few points off a real gridline doesn't create a
    spurious extra row/column."""
    values = sorted(values)
    out = []
    for v in values:
        if not out or v - out[-1] > tol:
            out.append(v)
    return out


def _table_settings_for(page):
    rects = page.rects
    horiz = _dedupe_lines([r["top"] for r in rects
                           if abs(r["top"] - r["bottom"]) < 1 and (r["x1"] - r["x0"]) > 100])
    vert = _dedupe_lines([r["x0"] for r in rects
                          if abs(r["x0"] - r["x1"]) < 1 and (r["bottom"] - r["top"]) > 5])
    if len(horiz) < 2 or len(vert) < 2:
        return None
    return {
        "vertical_strategy": "explicit",
        "horizontal_strategy": "explicit",
        "explicit_vertical_lines": vert,
        "explicit_horizontal_lines": horiz,
    }


def _extract_tables_from_page(page, section_markers):
    """Splits the page at section-header text before extracting tables, so
    tables with different column layouts are never merged together."""
    words = page.extract_words()
    page_top, page_bottom = page.bbox[1], page.bbox[3]

    split_points = []
    for marker in section_markers:
        first_word = marker.split()[0]
        for w in words:
            if w["text"] == first_word:
                split_points.append(w["top"] - 2)
    split_points = sorted(set(p for p in split_points if page_top < p < page_bottom))

    boundaries = [page_top] + split_points + [page_bottom]
    tables = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        if end - start < 5:
            continue
        crop = page.crop((page.bbox[0], start, page.bbox[2], end))
        settings = _table_settings_for(crop)
        if settings is None:
            continue
        for t in crop.extract_tables(settings):
            if t:
                tables.append(t)
    return tables


def extract_structured_tables(pdf_path, section_markers=None):
    if section_markers is None:
        section_markers = [
            "Agreement Description",
            "Samples and technical submission",
            "Agreement Approval",
        ]

    raw_tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            raw_tables.extend(_extract_tables_from_page(page, section_markers))

    # Stitch consecutive same-column-count fragments into one logical table
    grouped, current, current_ncols = [], [], None
    for t in raw_tables:
        ncols = len(t[0]) if t else 0
        if current_ncols is None or ncols == current_ncols:
            current.extend(t)
            current_ncols = ncols
        else:
            grouped.append(current)
            current, current_ncols = list(t), ncols
    if current:
        grouped.append(current)

    dataframes = []
    for group in grouped:
        header, rows = group[0], group[1:]
        if not rows:
            continue
        df = pd.DataFrame(rows, columns=header)
        # forward/back-fill the first column to handle merged/rowspan category cells
        df.iloc[:, 0] = df.iloc[:, 0].replace("", None).ffill().bfill()
        # a category name that wraps onto two lines (e.g. "Chemicals &" / "Insulation")
        # can land in two different rows instead of one merged cell — stitch it back together
        values = df.iloc[:, 0].tolist()
        for i in range(len(values) - 1):
            v = str(values[i]).strip()
            if v.endswith(("&", "-", "and")):
                merged = v + " " + str(values[i + 1]).strip()
                values[i] = merged
                values[i + 1] = merged
        df.iloc[:, 0] = values
        dataframes.append(df)
    return dataframes


data_tables = extract_structured_tables(DATA_PDF_PATH)

print(f"Found {len(data_tables)} tables in {DATA_PDF_PATH}:")
for i, df in enumerate(data_tables):
    print(f"  Table {i}: shape={df.shape}, columns={list(df.columns)}")


Inspect each table to confirm they look right, especially the main lead-time table
(it should be the largest one, with columns like Work Type / Limit / PO Creation / ...):
```python
data_tables[2].head(10)   # adjust the index to whichever table is the main one
```
If your PDF has a different layout than expected and a table looks wrong, the
`section_markers` list is the main thing to adjust — set it to whatever text appears
right before each distinct table/section in your document.


## 11. RAG answer function — Gemma answers using retrieved PDF context

In [ ]:
def rag_answer(question, chat_history=None):
    context_chunks = retrieve_relevant_chunks(question, top_k=3)
    context = "\n\n".join(context_chunks)

    system_prompt = (
        "You are an assistant that answers questions using ONLY the provided "
        "document context. If the answer isn't in the context, say you don't know."
    )

    full_prompt = f"""{system_prompt}

Context:
{context}

Question: {question}
Answer:"""

    chat = [{"role": "user", "content": full_prompt}]
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt")
    outputs = model.generate(input_ids=inputs.to(model.device), max_new_tokens=500)
    answer = tokenizer.decode(outputs[0]).split("model\n")[-1]
    return answer.strip()


## 12. Customer request logging

Every customer can send `/request`, and the bot will walk them through **name → item →
quantity**. After each item, the bot asks if they want to add another — so a single
`/request` session can collect **multiple items**. Once the customer says they're done,
the bot shows a **summary and asks for final confirmation** before saving anything.

Gemma matches each typed item against the real category names from PDF 2's main
lead-time table (so "ceramic tile stuff" gets matched to "Ceramic Tiles", etc.).

Once confirmed, each request session produces rows in **two Excel files**:

1. **The shared master log** (`customer_requests.xlsx`) — one growing file with every
   customer's requests (from every user, every session) appended as new rows,
   retrievable any time via `/requests`.
2. **A personal, persistent file for that customer** (`request_<chat_id>.xlsx`) —
   containing every request that specific Telegram user has ever made. If the same
   user runs `/request` again later in the chat, their new items are **appended** to
   this same file (not overwritten), and the updated file is sent back to them.


In [ ]:
def find_main_lead_time_table(tables):
    """Finds the main Work Type / Limit table among the extracted tables,
    regardless of its position in the list."""
    for df in tables:
        if "Work Type" in df.columns and "Limit" in df.columns:
            return df
    return None


main_lead_time_df = find_main_lead_time_table(data_tables)
known_categories = (
    main_lead_time_df["Work Type"].dropna().unique().tolist()
    if main_lead_time_df is not None else []
)

REQUESTS_LOG_PATH = "customer_requests.xlsx"
REQUEST_COLUMNS = [
    "Timestamp", "Telegram User", "Customer Name",
    "Requested Item", "Matched Category", "Quantity",
]

# In-memory master log for this session. Also saved to disk on every save so
# it survives even if the bot cell needs a moment between messages.
customer_requests_df = pd.DataFrame(columns=REQUEST_COLUMNS)


def match_category_with_gemma(item_text):
    if not known_categories:
        return "Unknown"

    categories_str = ", ".join(known_categories)
    prompt = (
        f"Known categories: {categories_str}\n\n"
        f'Match the customer\'s requested item to the SINGLE closest category '
        f'from the list above.\nRequested item: "{item_text}"\n'
        f"Respond with ONLY the matching category name, nothing else."
    )
    chat = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer.encode(prompt_text, add_special_tokens=False, return_tensors="pt")
    outputs = model.generate(input_ids=inputs.to(model.device), max_new_tokens=20)
    answer = tokenizer.decode(outputs[0]).split("model\n")[-1]
    return answer.strip()


def get_user_file_path(chat_id):
    """Each Telegram user gets one persistent Excel file that keeps growing
    across every /request session they ever run."""
    return f"request_{chat_id}.xlsx"


def save_customer_requests(message, name, items):
    """Saves a whole batch of items (collected over one /request session,
    possibly several items) as multiple rows.

    - Appends every row to the shared master log (all users, all time).
    - Appends every row to this specific user's own persistent file, loading
      whatever they already had on disk first so nothing is lost.

    Returns the path to the user's (now updated) personal file.
    """
    global customer_requests_df

    telegram_user = message.from_user.username or message.from_user.first_name
    now = pd.Timestamp.now()

    rows = []
    for entry in items:
        matched_category = match_category_with_gemma(entry["item"])
        rows.append({
            "Timestamp": now,
            "Telegram User": telegram_user,
            "Customer Name": name,
            "Requested Item": entry["item"],
            "Matched Category": matched_category,
            "Quantity": entry["quantity"],
        })
        # keep the matched category around so we can show it to the user too
        entry["matched_category"] = matched_category

    new_rows_df = pd.DataFrame(rows, columns=REQUEST_COLUMNS)

    # 1. Append to the shared master log
    customer_requests_df = pd.concat(
        [customer_requests_df, new_rows_df], ignore_index=True
    )
    customer_requests_df.to_excel(REQUESTS_LOG_PATH, index=False)

    # 2. Append to this user's own persistent file (load existing rows first,
    # so a later /request session adds to it instead of replacing it)
    user_path = get_user_file_path(message.chat.id)
    if os.path.exists(user_path):
        existing_df = pd.read_excel(user_path)
        user_df = pd.concat([existing_df, new_rows_df], ignore_index=True)
    else:
        user_df = new_rows_df
    user_df.to_excel(user_path, index=False)

    return user_path


## 13. Telegram bot

- Any normal message → answered via RAG using PDF 1.
- `/start` → welcome message.
- `/request` → walks the customer through name → item → quantity, matches their item
  to a known category using Gemma, and appends it to the shared customer requests log.
- `/requests` → sends back the current shared customer requests Excel file.

Running this cell blocks and keeps polling forever — this **is** your live bot.
Keep this Colab session open/connected while you want the bot to respond.


In [ ]:
import telebot

bot = telebot.TeleBot(telegram_token)


@bot.message_handler(commands=["start", "restart"])
def send_welcome(message):
    bot.reply_to(
        message,
        "Hi! Ask me anything about the document, or send /request to place a request."
    )


@bot.message_handler(commands=["request"])
def start_request(message):
    bot.reply_to(message, "Sure! What's your name?")
    bot.register_next_step_handler(message, request_get_name)


def request_get_name(message):
    name = message.text
    bot.reply_to(message, f"Thanks {name}! What item would you like to request?")
    bot.register_next_step_handler(message, request_get_item, name, [])


def request_get_item(message, name, items):
    item = message.text
    bot.reply_to(message, "Got it. How many units (quantity)?")
    bot.register_next_step_handler(message, request_get_quantity, name, items, item)


def request_get_quantity(message, name, items, item):
    quantity = message.text
    items.append({"item": item, "quantity": quantity})

    bot.reply_to(
        message,
        "Added ✅. Would you like to request another item? (yes/no)"
    )
    bot.register_next_step_handler(message, request_ask_more, name, items)


def request_ask_more(message, name, items):
    answer = message.text.strip().lower()

    if answer in ("yes", "y", "yeah", "yep"):
        bot.reply_to(message, "Sure, what's the next item?")
        bot.register_next_step_handler(message, request_get_item, name, items)
        return

    # No more items — show a summary and ask for final confirmation
    summary = "\n".join(
        f"{i+1}. {entry['item']} — qty {entry['quantity']}"
        for i, entry in enumerate(items)
    )
    bot.reply_to(
        message,
        "Here's what you've requested so far:\n"
        f"{summary}\n\n"
        "Is that everything? Shall I finalize and send you your request sheet? (yes/no)"
    )
    bot.register_next_step_handler(message, request_confirm, name, items)


def request_confirm(message, name, items):
    answer = message.text.strip().lower()

    if answer not in ("yes", "y", "yeah", "yep", "confirm"):
        # Not confirmed — let them keep adding items instead of losing progress
        bot.reply_to(message, "No problem, let's keep going. What's the next item?")
        bot.register_next_step_handler(message, request_get_item, name, items)
        return

    bot.reply_to(message, "Great, saving your request(s)...")

    user_path = save_customer_requests(message, name, items)

    summary = "\n".join(
        f"{i+1}. {entry['item']} → {entry.get('matched_category', 'Unknown')} "
        f"(qty {entry['quantity']})"
        for i, entry in enumerate(items)
    )
    bot.reply_to(
        message,
        "Request(s) logged! ✅\n"
        f"Name: {name}\n"
        f"{summary}"
    )

    with open(user_path, "rb") as f:
        bot.send_document(
            message.chat.id, f,
            caption="Here's your full request sheet (updated with today's items)."
        )


@bot.message_handler(commands=["requests"])
def send_requests_log(message):
    if customer_requests_df.empty:
        bot.reply_to(message, "No customer requests logged yet.")
        return
    with open(REQUESTS_LOG_PATH, "rb") as f:
        bot.send_document(message.chat.id, f)


@bot.message_handler()
def handle_message(message):
    question = message.text
    answer = rag_answer(question)
    bot.reply_to(message, answer)


print("Bot is running. Go to Telegram and message your bot.")
bot.infinity_polling()
